# Version 2 Retention Model Comparison

Checkpoint 41 compares fixed candidates on the 2024 temporal validation period. The 2025 test target remains reserved.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

comparison = pd.read_csv(PROCESSED_DIR / 'model_comparison_v2.csv')
grouped = pd.read_csv(PROCESSED_DIR / 'model_grouped_cv_metrics.csv')
pairwise = pd.read_csv(PROCESSED_DIR / 'model_pairwise_pr_auc_bootstrap.csv')
selection = pd.read_csv(PROCESSED_DIR / 'model_selection_decision_v2.csv')
predictions = pd.read_csv(PROCESSED_DIR / 'retention_validation_predictions_v2.csv')
checks = pd.read_csv(PROCESSED_DIR / 'model_comparison_validation.csv')

print(f'Candidate models: {len(comparison)}')
print(f'Validation employees per model: {predictions.employee_id.nunique():,}')
print(f'Integrity checks passed: {checks.status.eq("PASS").sum()}/{len(checks)}')

## Primary temporal validation

Models are fitted on 2023 and evaluated on 2024. PR-AUC is the primary ranking metric because attrition is uncommon.

In [ ]:
comparison[[
    'model', 'pr_auc', 'pr_auc_lower_95', 'pr_auc_upper_95',
    'roc_auc', 'brier_score', 'top_decile_lift',
    'top_decile_capture', 'selected_model'
]].sort_values('pr_auc', ascending=False)

In [ ]:
plot_data = comparison.sort_values('pr_auc')
lower_error = plot_data['pr_auc'] - plot_data['pr_auc_lower_95']
upper_error = plot_data['pr_auc_upper_95'] - plot_data['pr_auc']

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.errorbar(
    plot_data['pr_auc'], plot_data['model'],
    xerr=[lower_error, upper_error], fmt='o', capsize=5
)
ax.axvline(plot_data['positive_rate'].iloc[0], color='gray', linestyle='--', label='No-skill baseline')
ax.set_xlabel('Validation PR-AUC')
ax.set_title('Temporal validation PR-AUC with paired-bootstrap intervals')
ax.legend()
plt.tight_layout()
plt.show()

## Employee-grouped robustness

Each employee belongs to exactly one fold, so training and validation employees do not overlap within a fold.

In [ ]:
grouped.pivot(index='fold', columns='model', values='pr_auc')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
grouped.boxplot(column='pr_auc', by='model', ax=ax, grid=False)
ax.set_title('Employee-grouped PR-AUC across five folds')
ax.set_xlabel('')
ax.set_ylabel('PR-AUC')
plt.suptitle('')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Paired uncertainty and provisional selection

An interval containing zero means the observed PR-AUC difference is inconclusive. Simplicity is used only after performance and uncertainty are considered.

In [ ]:
pairwise

In [ ]:
selection[[
    'model', 'validation_pr_auc', 'gap_from_best_pr_auc',
    'paired_interval_includes_zero', 'within_practical_tolerance',
    'complexity_rank', 'selected_model'
]]

## Validation checks

The test population is counted from target-free assignments only. No test targets or probabilities are produced in this checkpoint.

In [ ]:
checks